# Assignment 9 — Vision-Based Cognitive Systems
**Course:** Cognitive Computing (UCS420)  
**Theme:** Landing AI — Vision-Based Cognitive System

## Tasks
1. Design a cognitive assistant for reading medicine-bottle labels using OCR.
2. Design an emotion-aware cognitive assistant using facial-expression detection.

**Note:** OCR output must be checked against the actual packaging. Facial-expression classification estimates visible expression patterns; it does not establish a person's true emotional state.

## 1. Install dependencies
Run once in a fresh Google Colab/Jupyter environment. Model downloads require internet access.

In [ ]:
%pip -q install easyocr opencv-python-headless pillow matplotlib transformers torch torchvision

## 2. Imports and setup

In [ ]:
import os, re, cv2, torch, easyocr, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Part A — Medicine-Bottle Label Reading Assistant

## 3. Upload a medicine-label image
Use a clear, well-lit image with the label facing the camera. In Colab, upload a file; outside Colab, provide its path.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded)) if uploaded else None
except Exception:
    IMAGE_PATH = input("Path to medicine-label image: ").strip() or None

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    label_image = Image.open(IMAGE_PATH).convert("RGB")
    display(label_image)
else:
    print("No image loaded. Upload/select an image before continuing.")

## 4. OCR text recognition
EasyOCR detects text regions and recognizes their contents. Its confidence score is not medical verification.

In [ ]:
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

def read_label_text(image_path, min_confidence=0.20):
    results = ocr_reader.readtext(image_path, detail=1, paragraph=False)
    return [{"text": text.strip(), "confidence": float(conf)}
            for bbox, text, conf in results
            if text.strip() and conf >= min_confidence]

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    ocr_lines = read_label_text(IMAGE_PATH)
    for item in ocr_lines:
        print(f'{item["text"]}  [confidence: {item["confidence"]:.2f}]')
else:
    ocr_lines = []

## 5. Extract likely medicine fields
A rule-based parser searches common label patterns. Results are candidates; packaging formats vary.

In [ ]:
def extract_medicine_fields(ocr_lines):
    full_text = "\n".join(x["text"] for x in ocr_lines)
    upper = full_text.upper()
    strength = re.search(r"\b\d+(?:\.\d+)?\s?(?:MG|MCG|G|ML|IU|%)\b", upper)
    expiry = re.search(r"\b(?:EXP|EXPIRY|EXPIRES?)\s*[:.-]?\s*((?:0?[1-9]|1[0-2])[/.-](?:20)?\d{2}|20\d{2}[-/]\d{1,2})", upper)
    batch = re.search(r"\b(?:BATCH(?:\s*NO)?|B\.?NO|LOT)\s*[:.#-]?\s*([A-Z0-9/-]{3,})", upper)
    mfg = re.search(r"\b(?:MFG|MFD|MANUFACTURED)\s*[:.-]?\s*([A-Z0-9 /.-]{4,})", upper)
    instructions = [x["text"] for x in ocr_lines
                    if re.search(r"\b(take|dosage|once|twice|daily|before|after|food|tablet|capsule)\b",
                                 x["text"], re.I)]
    excluded = ("BATCH", "LOT", "MFG", "MFD", "EXP", "MRP", "KEEP",
                "STORE", "DOSAGE", "TABLET", "CAPSULE")
    names = [x["text"] for x in ocr_lines if len(x["text"]) >= 4
             and re.search(r"[A-Za-z]", x["text"])
             and not any(w in x["text"].upper() for w in excluded)]
    return {
        "Possible medicine name": names[0] if names else "Not confidently identified",
        "Strength": strength.group(0) if strength else "Not detected",
        "Expiry date": expiry.group(1) if expiry else "Not detected",
        "Batch / lot number": batch.group(1) if batch else "Not detected",
        "Manufacturing information": mfg.group(1).strip() if mfg else "Not detected",
        "Possible dosage / instruction lines": " | ".join(instructions) if instructions else "Not detected",
        "Complete OCR text": full_text
    }

medicine_info = extract_medicine_fields(ocr_lines) if ocr_lines else {}
for field, value in medicine_info.items():
    print(f"\n{field}:\n{value}")

## 6. Assistant response
The assistant reports candidate fields and asks the user to verify them. It does not recommend a dose.

In [ ]:
def medicine_assistant_response(info):
    if not info:
        return "Upload a label image and run OCR to obtain a result."
    keys = ["Possible medicine name", "Strength", "Expiry date",
            "Batch / lot number", "Manufacturing information",
            "Possible dosage / instruction lines"]
    output = ["### Medicine-label reading result"]
    output += [f'- **{k}:** {info.get(k, "Not detected")}' for k in keys]
    output.append("\n**Verification required:** Check every field on the original package. "
                  "Do not use OCR output to decide whether or how much medicine to take.")
    return "\n".join(output)

display(Markdown(medicine_assistant_response(medicine_info)))

## 7. Visualize OCR regions

In [ ]:
def show_ocr_boxes(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print("Could not open image."); return
    for bbox, text, confidence in ocr_reader.readtext(image_path, detail=1):
        cv2.polylines(image, [np.array(bbox, dtype=np.int32)], True, (0, 200, 0), 2)
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off"); plt.title("OCR-detected text regions"); plt.show()

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    show_ocr_boxes(IMAGE_PATH)

# Part B — Emotion-Aware Assistant Using Facial Expression Detection

## 8. Load a pretrained model
The Vision Transformer model below predicts expression categories learned from its training data. The first run downloads model weights.

In [ ]:
from transformers import pipeline

emotion_classifier = pipeline(
    "image-classification",
    model="trpakov/vit-face-expression",
    device=0 if DEVICE == "cuda" else -1
)
print("Expression classifier loaded.")

## 9. Upload a face image and detect a face
Use a front-facing, well-lit image. If multiple faces appear, this implementation selects the largest detected face.

In [ ]:
try:
    from google.colab import files
    print("Upload a face image.")
    uploaded_face = files.upload()
    FACE_PATH = next(iter(uploaded_face)) if uploaded_face else None
except Exception:
    FACE_PATH = input("Path to face image: ").strip() or None

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def detect_face_crop(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Image could not be opened; check its path.")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(50, 50))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    pad = int(0.15 * max(w, h))
    x1, y1 = max(0, x-pad), max(0, y-pad)
    x2, y2 = min(image.shape[1], x+w+pad), min(image.shape[0], y+h+pad)
    return Image.fromarray(cv2.cvtColor(image[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))

face_crop = detect_face_crop(FACE_PATH) if FACE_PATH and os.path.exists(FACE_PATH) else None
if face_crop is not None:
    display(face_crop)
else:
    print("No face detected or no image loaded. Try a clearer front-facing image.")

## 10. Classify the facial expression

In [ ]:
def predict_expression(face_image):
    return emotion_classifier(face_image)

emotion_results = predict_expression(face_crop) if face_crop is not None else []
for result in emotion_results:
    print(f'{result["label"]}: {result["score"]:.3f}')

## 11. Emotion-aware assistant response
The response describes the model's leading facial-expression category, not the person's internal feelings.

In [ ]:
def emotion_assistant_response(results):
    if not results:
        return "No prediction available. Provide a clear image containing a visible face."
    top = results[0]
    label = top["label"].replace("_", " ").title()
    score = top["score"]
    certainty = ("relatively low" if score < 0.45 else
                 "moderate" if score < 0.70 else "comparatively high")
    return (f"**Predicted expression category:** {label}\n\n"
            f"**Model score:** {score:.3f}\n\n"
            f"The model's confidence is {certainty}. This is only a visual prediction; "
            "it does not prove how the person actually feels. Do not use it for consequential decisions.")

display(Markdown(emotion_assistant_response(emotion_results)))

## 12. Plot expression scores

In [ ]:
if emotion_results:
    labels = [r["label"].replace("_", " ").title() for r in emotion_results]
    scores = [r["score"] for r in emotion_results]
    plt.figure(figsize=(9, 4))
    plt.bar(labels, scores)
    plt.ylabel("Model score"); plt.xlabel("Expression category")
    plt.title("Facial-expression classification scores")
    plt.xticks(rotation=25, ha="right"); plt.ylim(0, 1)
    plt.tight_layout(); plt.show()
else:
    print("No predictions available to plot.")

# 13. Results, observations and conclusion

## Results
- **Medicine-label assistant:** OCR recognizes text from the bottle image. A rule-based extractor identifies likely medicine name, strength, expiry date, batch/lot number and instruction lines.
- **Emotion-aware assistant:** a face detector locates a face, and a pretrained image classifier returns ranked facial-expression categories and scores.

## Observations
1. Blur, glare, curved packaging, small fonts and unusual layouts reduce OCR accuracy.
2. OCR confidence is not factual correctness; label fields must be checked on the package.
3. Face detection may fail because of poor lighting, occlusion, side profiles or small faces.
4. Expression classification can be affected by pose, lighting, dataset bias and individual differences.

## Conclusion
Both tasks follow a vision-based cognitive pipeline: **image acquisition → preprocessing/detection → model inference → information extraction or classification → user-facing response**. The medicine assistant converts label images into structured candidate information, while the emotion-aware assistant classifies visible facial-expression patterns. Both systems require human oversight.

## References
- EasyOCR: https://github.com/JaidedAI/EasyOCR
- OpenCV: https://docs.opencv.org/
- Model card: https://huggingface.co/trpakov/vit-face-expression
- Transformers: https://huggingface.co/docs/transformers

**End of Assignment 9.**